# 18 — All-six-variable baseline v1

Train the all-six-variable small-CNN baseline on the canonical
internal training split. Select the checkpoint and operating
threshold using validation only. The official test split
remains untouched.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.7"
VALIDATION_FRACTION = 0.20
VALIDATION_SEED = 20260913
TRAINING_SEED = 20260913
YEARS = tuple(range(2013, 2023))
VARIABLES = (
    "DBZ",
    "KDP",
    "RHOHV",
    "VEL",
    "WIDTH",
    "ZDR",
)
CHANNEL_ORDER = tuple(
    f"{variable}_sweep_{sweep}"
    for variable in VARIABLES
    for sweep in range(2)
)
CHANNEL_COUNT = len(CHANNEL_ORDER)

FILE_BATCH_SIZE = 16
NUM_WORKERS = 12
MAX_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 5
LEARNING_RATE = 1e-3
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
MINIMUM_LEARNING_RATE = 1e-5

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        f"tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = (
    BACKUP_ROOT / "manifests"
)
NORMALIZATION_PATH = (
    BACKUP_ROOT
    / "experiments"
    / "all_year_all6_baseline_v1"
    / "normalization.json"
)
EXPERIMENT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "all_year_all6_baseline_v1"
)
LAST_CHECKPOINT_PATH = (
    EXPERIMENT_DIRECTORY
    / "last_checkpoint.pt"
)
BEST_CHECKPOINT_PATH = (
    EXPERIMENT_DIRECTORY
    / "best_model.pt"
)
HISTORY_PATH = (
    EXPERIMENT_DIRECTORY
    / "training_history.csv"
)
VALIDATION_METRICS_PATH = (
    EXPERIMENT_DIRECTORY
    / "validation_metrics.json"
)
EXTRACTION_ROOT = Path(
    "/content/tornet_all_year_baseline"
)

for path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    NORMALIZATION_PATH,
):
    if not path.exists():
        raise FileNotFoundError(path)

if VALIDATION_METRICS_PATH.exists():
    raise FileExistsError(
        "Experiment already completed: "
        f"{VALIDATION_METRICS_PATH}"
    )

EXPERIMENT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

print("package:", PACKAGE_PATH)
print(
    "normalization:",
    NORMALIZATION_PATH,
)
print(
    "experiment:",
    EXPERIMENT_DIRECTORY,
)

package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.7-py3-none-any.whl
normalization: /content/drive/MyDrive/TorNet_Backup/experiments/all_year_all6_baseline_v1/normalization.json
experiment: /content/drive/MyDrive/TorNet_Backup/experiments/all_year_all6_baseline_v1


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
        "scikit-learn>=1.5",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.7-py3-none-any.whl'], returncode=0)

In [4]:
import datetime
import json
import random
import shutil
import tarfile
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import (
    DataLoader,
    Dataset,
)

import tornado_detection
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
    read_netcdf_file,
)

if (
    tornado_detection.__version__
    != PACKAGE_VERSION
):
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select an A100 GPU runtime "
        "and run all cells"
    )

random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
torch.cuda.manual_seed_all(
    TRAINING_SEED
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda")

normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)

expected_normalization = {
    "package_version": PACKAGE_VERSION,
    "years": list(YEARS),
    "model_split": "train",
    "validation_fraction": (
        VALIDATION_FRACTION
    ),
    "validation_seed": (
        VALIDATION_SEED
    ),
    "training_frame_count": 547_672,
    "training_file_count": 136_918,
    "training_positive_frame_count": (
        18_230
    ),
    "variables": list(VARIABLES),
    "channel_order": list(
        CHANNEL_ORDER
    ),
    "tensor_shape": [
        120,
        240,
        CHANNEL_COUNT,
    ],
}

mismatches = {
    key: {
        "expected": expected,
        "actual": normalization.get(key),
    }
    for key, expected
    in expected_normalization.items()
    if normalization.get(key) != expected
}

if mismatches:
    raise AssertionError(
        "Normalization provenance "
        f"mismatch: {mismatches}"
    )

means = np.asarray(
    normalization["means"],
    dtype=np.float32,
)
stds = np.asarray(
    normalization[
        "standard_deviations"
    ],
    dtype=np.float32,
)

assert means.shape == (
    CHANNEL_COUNT,
)
assert stds.shape == (
    CHANNEL_COUNT,
)

canonical = (
    load_canonical_frame_index(
        MANIFESTS_ROOT
    )
)
assigned = assign_model_splits(
    canonical,
    validation_fraction=(
        VALIDATION_FRACTION
    ),
    seed=VALIDATION_SEED,
)

train_rows = (
    assigned.loc[
        assigned["model_split"].eq(
            "train"
        )
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ]
    )
)
validation_rows = (
    assigned.loc[
        assigned["model_split"].eq(
            "validation"
        )
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ]
    )
)
test_rows = assigned.loc[
    assigned["model_split"].eq(
        "test"
    )
]

assert len(train_rows) == 547_672
assert int(
    train_rows["frame_label"].sum()
) == 18_230
assert len(validation_rows) == 138_892
assert int(
    validation_rows[
        "frame_label"
    ].sum()
) == 4_200
assert len(test_rows) == 125_868
assert int(
    test_rows["frame_label"].sum()
) == 3_909

for rows in (
    train_rows,
    validation_rows,
):
    counts = rows.groupby(
        "archive_member"
    ).size()

    if not counts.eq(4).all():
        raise AssertionError(
            "Every selected file must "
            "contain four frames"
        )


def build_records(rows):
    return [
        (
            member,
            group["frame_label"]
            .astype(np.uint8)
            .to_numpy(),
        )
        for member, group
        in rows.groupby(
            "archive_member",
            sort=True,
        )
    ]


train_records = build_records(
    train_rows
)
validation_records = build_records(
    validation_rows
)

required_by_year = {
    year: set(
        assigned.loc[
            assigned["year"].eq(year)
            & assigned[
                "model_split"
            ].isin(
                [
                    "train",
                    "validation",
                ]
            ),
            "archive_member",
        ].unique()
    )
    for year in YEARS
}

print(
    "GPU:",
    torch.cuda.get_device_name(0),
)
print(
    "train files/frames:",
    len(train_records),
    len(train_rows),
)
print(
    "validation files/frames:",
    len(validation_records),
    len(validation_rows),
)
print(
    "official test untouched:",
    len(test_rows),
)

GPU: NVIDIA A100-SXM4-40GB
train files/frames: 136918 547672
validation files/frames: 34723 138892
official test untouched: 125868


In [5]:
if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

staging = []

for year in YEARS:
    drive_archive = (
        BACKUP_ROOT
        / f"tornet_{year}.tar.gz"
    )
    local_archive = Path(
        f"/content/tornet_{year}.tar.gz"
    )

    if not drive_archive.is_file():
        raise FileNotFoundError(
            drive_archive
        )

    if local_archive.exists():
        local_archive.unlink()

    copy_started = (
        time.perf_counter()
    )

    shutil.copyfile(
        drive_archive,
        local_archive,
    )

    copy_seconds = (
        time.perf_counter()
        - copy_started
    )

    required = required_by_year[year]
    extracted = set()
    extract_started = (
        time.perf_counter()
    )

    with tarfile.open(
        local_archive,
        mode="r:gz",
    ) as archive:
        for member in archive:
            if (
                not member.isfile()
                or member.name
                not in required
            ):
                continue

            destination = (
                EXTRACTION_ROOT
                / member.name
            )
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            source_file = (
                archive.extractfile(
                    member
                )
            )

            if source_file is None:
                raise RuntimeError(
                    member.name
                )

            with (
                source_file,
                destination.open("wb")
                as output_file,
            ):
                shutil.copyfileobj(
                    source_file,
                    output_file,
                    length=1024 * 1024,
                )

            extracted.add(
                member.name
            )

    extraction_seconds = (
        time.perf_counter()
        - extract_started
    )
    missing = required - extracted

    if missing:
        raise RuntimeError(
            f"Year {year} missing: "
            f"{sorted(missing)[:10]}"
        )

    local_archive.unlink()

    staging.append(
        {
            "year": year,
            "file_count": len(
                extracted
            ),
            "copy_seconds": (
                copy_seconds
            ),
            "extraction_seconds": (
                extraction_seconds
            ),
        }
    )

    print(
        f"year={year} "
        f"staged_files="
        f"{len(extracted):,} "
        f"copy={copy_seconds:.1f}s "
        f"extract="
        f"{extraction_seconds:.1f}s"
    )

print(
    "total staged files:",
    sum(
        row["file_count"]
        for row in staging
    ),
)

staged_bytes = sum(
    path.stat().st_size
    for path
    in EXTRACTION_ROOT.rglob("*.nc")
)

print(
    "staged GiB:",
    round(
        staged_bytes / 1024**3,
        3,
    ),
)

year=2013 staged_files=3,498 copy=106.7s extract=16.4s
year=2014 staged_files=17,446 copy=443.5s extract=57.9s
year=2015 staged_files=19,721 copy=337.9s extract=89.0s
year=2016 staged_files=18,791 copy=528.5s extract=75.0s
year=2017 staged_files=16,927 copy=331.7s extract=62.8s
year=2018 staged_files=15,353 copy=441.2s extract=44.3s
year=2019 staged_files=20,588 copy=515.5s extract=93.1s
year=2020 staged_files=18,147 copy=431.7s extract=81.9s
year=2021 staged_files=19,032 copy=599.1s extract=89.6s
year=2022 staged_files=22,138 copy=666.3s extract=95.3s
total staged files: 171641
staged GiB: 128.017


In [6]:
class FileDataset(Dataset):
    def __init__(
        self,
        file_records,
        root,
        channel_means,
        channel_stds,
    ):
        self.records = file_records
        self.root = root
        self.means = (
            channel_means.reshape(
                1,
                1,
                1,
                CHANNEL_COUNT,
            )
        )
        self.stds = (
            channel_stds.reshape(
                1,
                1,
                1,
                CHANNEL_COUNT,
            )
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        (
            member,
            expected_labels,
        ) = self.records[index]

        result = read_netcdf_file(
            self.root / member,
            variables=VARIABLES,
        )

        np.testing.assert_array_equal(
            result.labels,
            expected_labels,
        )

        values = np.nan_to_num(
            (
                result.values
                - self.means
            )
            / self.stds,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(
            np.float32,
            copy=False,
        )

        inputs = (
            torch.from_numpy(values)
            .permute(0, 3, 1, 2)
            .contiguous()
        )
        labels = torch.from_numpy(
            result.labels.astype(
                np.float32,
                copy=False,
            )
        ).reshape(4, 1)

        return inputs, labels


class RadarBaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                CHANNEL_COUNT,
                16,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                16,
                32,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                32,
                64,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                64,
                128,
                3,
                padding=1,
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        self.classifier = (
            nn.Sequential(
                nn.Flatten(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, 1),
            )
        )

    def forward(self, inputs):
        return self.classifier(
            self.features(inputs)
        )


train_dataset = FileDataset(
    train_records,
    EXTRACTION_ROOT,
    means,
    stds,
)
validation_dataset = FileDataset(
    validation_records,
    EXTRACTION_ROOT,
    means,
    stds,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=FILE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

model = RadarBaselineCNN().to(
    device
)

train_positive_count = int(
    train_rows["frame_label"].sum()
)
train_negative_count = (
    len(train_rows)
    - train_positive_count
)
positive_weight = torch.tensor(
    [
        train_negative_count
        / train_positive_count
    ],
    device=device,
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=positive_weight
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)
scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MINIMUM_LEARNING_RATE,
    )
)

print(
    "parameters:",
    f"{sum(parameter.numel() for parameter in model.parameters()):,}",
)
print(
    "positive weight:",
    float(positive_weight.cpu()),
)

parameters: 107,537
positive weight: 29.042346954345703


In [7]:
def flatten_batch(
    inputs,
    targets,
):
    files, frames = inputs.shape[:2]

    return (
        inputs.reshape(
            files * frames,
            CHANNEL_COUNT,
            120,
            240,
        ),
        targets.reshape(
            files * frames,
            1,
        ),
    )


def evaluate(loader):
    model.eval()

    labels = []
    probabilities = []
    loss_sum = 0.0
    count = 0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = (
                flatten_batch(
                    inputs,
                    targets,
                )
            )
            inputs = inputs.to(
                device,
                non_blocking=True,
            )
            targets = targets.to(
                device,
                non_blocking=True,
            )

            logits = model(inputs)
            loss = criterion(
                logits,
                targets,
            )
            size = int(
                targets.shape[0]
            )

            loss_sum += (
                float(loss.cpu())
                * size
            )
            count += size

            labels.extend(
                targets.cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )
            probabilities.extend(
                logits.sigmoid()
                .cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    return (
        loss_sum / count,
        float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        labels,
        probabilities,
    )


history = []
best_pr_auc = -1.0
best_epoch = 0
epochs_without_improvement = 0
start_epoch = 1

if LAST_CHECKPOINT_PATH.exists():
    state = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        state["model_state_dict"]
    )
    optimizer.load_state_dict(
        state["optimizer_state_dict"]
    )
    scheduler.load_state_dict(
        state["scheduler_state_dict"]
    )

    history = state["history"]
    best_pr_auc = float(
        state["best_pr_auc"]
    )
    best_epoch = int(
        state["best_epoch"]
    )
    epochs_without_improvement = int(
        state[
            "epochs_without_improvement"
        ]
    )
    start_epoch = (
        int(state["epoch"]) + 1
    )

    print(
        "resuming at epoch:",
        start_epoch,
    )

for epoch in range(
    start_epoch,
    MAX_EPOCHS + 1,
):
    generator = (
        torch.Generator()
        .manual_seed(
            TRAINING_SEED + epoch
        )
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=FILE_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
        generator=generator,
    )

    model.train()
    train_loss_sum = 0.0
    train_count = 0
    started = time.perf_counter()

    for inputs, targets in train_loader:
        inputs, targets = flatten_batch(
            inputs,
            targets,
        )
        inputs = inputs.to(
            device,
            non_blocking=True,
        )
        targets = targets.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )
        logits = model(inputs)
        loss = criterion(
            logits,
            targets,
        )
        loss.backward()
        optimizer.step()

        size = int(targets.shape[0])
        train_loss_sum += (
            float(
                loss.detach().cpu()
            )
            * size
        )
        train_count += size

    (
        validation_loss,
        validation_pr_auc,
        validation_roc_auc,
        _,
        _,
    ) = evaluate(validation_loader)

    scheduler.step(
        validation_pr_auc
    )

    elapsed = (
        time.perf_counter()
        - started
    )
    improved = (
        validation_pr_auc
        > best_pr_auc
    )

    if improved:
        best_pr_auc = (
            validation_pr_auc
        )
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": (
                    model.state_dict()
                ),
                "validation_pr_auc": (
                    validation_pr_auc
                ),
            },
            BEST_CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    row = {
        "epoch": epoch,
        "train_loss": (
            train_loss_sum
            / train_count
        ),
        "validation_loss": (
            validation_loss
        ),
        "validation_pr_auc": (
            validation_pr_auc
        ),
        "validation_roc_auc": (
            validation_roc_auc
        ),
        "learning_rate": (
            optimizer.param_groups[0][
                "lr"
            ]
        ),
        "epoch_seconds": elapsed,
        "improved": improved,
    }
    history.append(row)

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": (
                model.state_dict()
            ),
            "optimizer_state_dict": (
                optimizer.state_dict()
            ),
            "scheduler_state_dict": (
                scheduler.state_dict()
            ),
            "history": history,
            "best_pr_auc": (
                best_pr_auc
            ),
            "best_epoch": best_epoch,
            "epochs_without_improvement": (
                epochs_without_improvement
            ),
        },
        LAST_CHECKPOINT_PATH,
    )

    print(
        f"epoch={epoch:02d} "
        f"train_loss="
        f"{row['train_loss']:.6f} "
        f"val_loss="
        f"{validation_loss:.6f} "
        f"val_pr_auc="
        f"{validation_pr_auc:.6f} "
        f"val_roc_auc="
        f"{validation_roc_auc:.6f} "
        f"lr="
        f"{row['learning_rate']:.6g} "
        f"seconds={elapsed:.1f} "
        f"best={best_epoch}"
    )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print("early stopping")
        break

resuming at epoch: 12
epoch=12 train_loss=0.725527 val_loss=0.730201 val_pr_auc=0.412093 val_roc_auc=0.905056 lr=0.001 seconds=2110.9 best=12
epoch=13 train_loss=0.712693 val_loss=0.750746 val_pr_auc=0.402257 val_roc_auc=0.901500 lr=0.001 seconds=2136.2 best=12
epoch=14 train_loss=0.703280 val_loss=0.739225 val_pr_auc=0.392784 val_roc_auc=0.908060 lr=0.001 seconds=2128.9 best=12
epoch=15 train_loss=0.689185 val_loss=0.725318 val_pr_auc=0.411504 val_roc_auc=0.904735 lr=0.0005 seconds=2122.2 best=12
epoch=16 train_loss=0.650365 val_loss=0.738460 val_pr_auc=0.433641 val_roc_auc=0.908997 lr=0.0005 seconds=2112.5 best=16
epoch=17 train_loss=0.636556 val_loss=0.728129 val_pr_auc=0.428400 val_roc_auc=0.910190 lr=0.0005 seconds=2096.8 best=16
epoch=18 train_loss=0.628806 val_loss=0.729656 val_pr_auc=0.436165 val_roc_auc=0.912577 lr=0.0005 seconds=2110.4 best=18
epoch=19 train_loss=0.621181 val_loss=0.724210 val_pr_auc=0.430347 val_roc_auc=0.913984 lr=0.0005 seconds=2117.9 best=18
epoch=20 trai

In [8]:
best_state = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)
model.load_state_dict(
    best_state["model_state_dict"]
)

(
    validation_loss,
    validation_pr_auc,
    validation_roc_auc,
    labels,
    probabilities,
) = evaluate(validation_loader)

(
    precision_values,
    recall_values,
    thresholds,
) = precision_recall_curve(
    labels,
    probabilities,
)

f1_values = (
    2
    * precision_values[:-1]
    * recall_values[:-1]
    / np.maximum(
        (
            precision_values[:-1]
            + recall_values[:-1]
        ),
        1e-12,
    )
)
threshold_index = int(
    np.argmax(f1_values)
)
threshold = float(
    thresholds[threshold_index]
)
predictions = (
    probabilities >= threshold
).astype(np.int64)

precision = float(
    precision_score(
        labels,
        predictions,
        zero_division=0,
    )
)
recall = float(
    recall_score(
        labels,
        predictions,
        zero_division=0,
    )
)
f1 = float(
    2
    * precision
    * recall
    / max(
        precision + recall,
        1e-12,
    )
)

matrix = confusion_matrix(
    labels,
    predictions,
    labels=[0, 1],
)
(
    true_negative,
    false_positive,
    false_negative,
    true_positive,
) = [
    int(value)
    for value in matrix.ravel()
]

metrics = {
    "artifact_kind": (
        "all_year_all6_baseline_"
        "validation_metrics"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": (
        PACKAGE_VERSION
    ),
    "variables": list(VARIABLES),
    "channel_order": list(
        CHANNEL_ORDER
    ),
    "channel_count": CHANNEL_COUNT,
    "official_test_evaluated": False,
    "checkpoint_epoch": int(
        best_state["epoch"]
    ),
    "validation_frame_count": int(
        len(labels)
    ),
    "validation_positive_count": int(
        labels.sum()
    ),
    "validation_pr_auc": (
        validation_pr_auc
    ),
    "validation_roc_auc": (
        validation_roc_auc
    ),
    "validation_loss": (
        validation_loss
    ),
    "selected_threshold": threshold,
    "threshold_selection": (
        "maximum validation F1"
    ),
    "validation_precision": (
        precision
    ),
    "validation_recall": recall,
    "validation_f1": f1,
    "true_negative": true_negative,
    "false_positive": false_positive,
    "false_negative": false_negative,
    "true_positive": true_positive,
    "normalization_path": str(
        NORMALIZATION_PATH
    ),
    "history_path": str(
        HISTORY_PATH
    ),
    "staging": staging,
}

VALIDATION_METRICS_PATH.write_text(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
)
print(
    "wrote:",
    VALIDATION_METRICS_PATH,
)

{
  "artifact_kind": "all_year_all6_baseline_validation_metrics",
  "channel_count": 12,
  "channel_order": [
    "DBZ_sweep_0",
    "DBZ_sweep_1",
    "KDP_sweep_0",
    "KDP_sweep_1",
    "RHOHV_sweep_0",
    "RHOHV_sweep_1",
    "VEL_sweep_0",
    "VEL_sweep_1",
    "WIDTH_sweep_0",
    "WIDTH_sweep_1",
    "ZDR_sweep_0",
    "ZDR_sweep_1"
  ],
  "checkpoint_epoch": 20,
  "created_at_utc": "2026-09-22T12:54:40.278519+00:00",
  "false_negative": 2027,
  "false_positive": 2896,
  "history_path": "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_all6_baseline_v1/training_history.csv",
  "normalization_path": "/content/drive/MyDrive/TorNet_Backup/experiments/all_year_all6_baseline_v1/normalization.json",
  "official_test_evaluated": false,
  "package_version": "0.1.7",
  "selected_threshold": 0.8699302077293396,
  "staging": [
    {
      "copy_seconds": 106.73396335899815,
      "extraction_seconds": 16.395550168999762,
      "file_count": 3498,
      "year": 2013
    },
    {

In [9]:
shutil.rmtree(EXTRACTION_ROOT)

assert not EXTRACTION_ROOT.exists()
assert BEST_CHECKPOINT_PATH.is_file()
assert HISTORY_PATH.is_file()
assert VALIDATION_METRICS_PATH.is_file()

print(
    "Removed all Colab-local "
    "training data"
)
print(
    "Preserved experiment:",
    EXPERIMENT_DIRECTORY,
)

Removed all Colab-local training data
Preserved experiment: /content/drive/MyDrive/TorNet_Backup/experiments/all_year_all6_baseline_v1
